# OpenContrails dataset documentation
 
The satellite images are originally obtained from the [GOES-16 Advanced Baseline Imager (ABI)](https://www.goes-r.gov/spacesegment/abi.html), which is publicly available on [Google Cloud Storage](https://console.cloud.google.com/storage/browser/gcp-public-data-goes-16/). The original full-disk images were reprojected using bilinear resampling to generate a local scene image. Because contrails are easier to identify with temporal context, a sequence of images at 10-minute intervals are provided. Each example contains exactly one labeled frame.

Learn more about the dataset from our preprint: [OpenContrails: Benchmarking Contrail Detection on GOES-16 ABI](https://arxiv.org/abs/2304.02122).

## Files
In each subdirectory named by `{record_id}`, binary files in numpy `.npy` format that corresponds to a single example are provided:

* **`band_{08-16}.npy`**: array with size of `H x W x T`, where `T = n_times_before + n_times_after + 1`, representing the number of images in the sequence. There are `n_times_before` and `n_times_after` images before and after the labeled frame respectively. In our dataset all examples have  `n_times_before=4` and `n_times_after=3`. Each band represents an infrared channel at different wavelengths and is converted to brightness temperatures based on the calibration parameters. The number in the filename corresponds to the GOES-16 ABI band number. Details of the ABI bands can be found [here](https://www.goes-r.gov/mission/ABI-bands-quick-info.html).
* **`human_individual_masks.npy`**: array with size of `H x W x 1 x R`. Each example is labeled by `R` individual human labelers. `R` is not the same for all samples. The labeled masks have value either 0 or 1 and correspond to the `(n_times_before+1)`-th image in `band_{08-16}.npy`. They are available only in the training set.
* **`human_pixel_masks.npy`**: array with size of `H x W x 1` containing the  binary groundtruth. A pixel is regarded as contrail pixel in evaluation if it is labeled as contrail by more than half of the labelers. 

**`{train/validation}_metadata.json`**: contains the timestamps and the projection parameters to reproduce the satellite images.


In [ ]:
import os
import numpy as np
from matplotlib import animation
import matplotlib.pyplot as plt
from IPython import display

In [ ]:
BASE_DIR = '/kaggle/input/google-research-identify-contrails-reduce-global-warming/train'
N_TIMES_BEFORE = 4
record_id = '1704010292581573769'

def get_human_mask(record_id_cur=record_id):
    with open(os.path.join(BASE_DIR, record_id_cur, 'human_pixel_masks.npy'), 'rb') as f:
        human_pixel_mask = np.load(f)
    return (human_pixel_mask)

with open(os.path.join(BASE_DIR, record_id, 'human_pixel_masks.npy'), 'rb') as f:
    human_pixel_mask = np.load(f)
with open(os.path.join(BASE_DIR, record_id, 'human_individual_masks.npy'), 'rb') as f:
    human_individual_mask = np.load(f)

In [ ]:
def band(number,record_id_cur=record_id):
    with open(os.path.join(BASE_DIR, record_id_cur, f'band_{number:02}.npy'), 'rb') as f:
        band_i = np.load(f)
    return (band_i)

In [ ]:

def get_folder_names(directory_path):
    folder_names = [f for f in os.listdir(directory_path) if os.path.isdir(os.path.join(directory_path, f))]
    return folder_names

train_data_names = get_folder_names(BASE_DIR)
train_data_names[:5]

### Combine bands into a false color image
In order to view contrails in GOES, we use the "ash" color scheme. This color scheme was originally developed for viewing volcanic ash in the atmosphere but is also useful for viewing thin cirrus, including contrails. In this color scheme, contrails appear in the image as dark blue.

Note that we use a modified version of the ash color scheme here, developed by Kulik et al., which uses slightly different bands and bounds tuned for contrails.

References:
 - Ash Color Scheme (page 7): https://eumetrain.org/sites/default/files/2020-05/RGB_recipes.pdf

In [ ]:
#get reference pictures (REFERENCE CODE)
_T11_BOUNDS = (243, 303)
_CLOUD_TOP_TDIFF_BOUNDS = (-4, 5)
_TDIFF_BOUNDS = (-4, 2)

def normalize_range(data, bounds):
    """Maps data to the range [0, 1]."""
    return (data - bounds[0]) / (bounds[1] - bounds[0])

lam=np.array([0.,0.47,0.64,0.86,1.37,1.6,2.2,3.9,6.2,6.9,7.3,8.4,9.6,10.3,11.2,12.3,13.3])
def spektral_index(i1,i2,record_id_cur=record_id):
    r = normalize_range(np.log(band(i1,record_id_cur=record_id_cur)/band(i2,record_id_cur=record_id_cur))/np.log(lam[i1]/lam[i2]), _TDIFF_BOUNDS)
    false_color = np.clip(np.stack([r], axis=2), 0, 1)
    img = false_color[..., N_TIMES_BEFORE]
    return(img)

r1 = normalize_range(band(15) - band(14), _TDIFF_BOUNDS)
g1 = normalize_range(band(14) - band(11), _CLOUD_TOP_TDIFF_BOUNDS)
b1 = normalize_range(band(14), _T11_BOUNDS)
false_color1 = np.clip(np.stack([r1, g1, b1], axis=2), 0, 1)
img1=false_color1[..., N_TIMES_BEFORE]

#g = normalize_range(band14 - band11, _CLOUD_TOP_TDIFF_BOUNDS)
#b = normalize_range(band14, _T11_BOUNDS)

In [ ]:
current_mask = (human_pixel_mask==1)

current_mask.shape
mask_3d = np.zeros((256, 256, 3),dtype=bool)
mask_3d[:,:,0] = current_mask[:,:,0]
mask_3d[:,:,1] = current_mask[:,:,0]
mask_3d[:,:,2] = current_mask[:,:,0]

## Visualize data

In [ ]:
for i in range(8,16):
    img=spektral_index(i,i+1)
    plt.figure(figsize=(20, 5))
    
    ax = plt.subplot(1, 4, 1)
    ax.imshow(img1)
    ax.set_title(f"Reference")
    
    ax = plt.subplot(1, 4, 2)
    ax.imshow(img)
    ax.set_title(f'Spektral index {i}-{i+1}')
    
    current_mask = (human_pixel_mask==1)
    masked_img = np.copy(img)
    masked_img[current_mask] = None
    
    masked_img1 = np.copy(img1)
    masked_img1[mask_3d] = None

    ax = plt.subplot(1, 4, 3)
    ax.imshow(masked_img1, interpolation='none')
    ax.set_title('Contrail mask')

    ax = plt.subplot(1, 4, 4)
    ax.imshow(masked_img, interpolation='none')
    ax.set_title('Contrail mask')


    plt.show()

In [ ]:
#taking the best pictures and averaging them
diff_img=(spektral_index(13,14)+spektral_index(13,15))/2
plt.figure(figsize=(18, 6))

ax = plt.subplot(1, 3, 1)
ax.imshow(diff_img)
ax.set_title(f"Diff. Image")

ax = plt.subplot(1, 3, 2)
ax.imshow(spektral_index(13,14))
ax.set_title(f"13-14")

ax = plt.subplot(1, 3, 3)
ax.imshow(spektral_index(13,15))
ax.set_title(f"13-15")

plt.show()

In [ ]:
#histograms for spectral index 13-14 (nine random images)
print(len(train_data_names))
idx_train_data_list = [535,567,6576,168,377,118,587,500,790]
for idx_train_data in idx_train_data_list:
    record_id_cur = train_data_names[idx_train_data]
    for i in range(13,14):
        for j in range(i+1,i+2):

            human_pixel_mask = get_human_mask(record_id_cur)
            img_test = spektral_index(i,j,record_id_cur)
            current_mask = (human_pixel_mask==1)
            masked_img = np.copy(img_test)
            pos=masked_img[current_mask]
            neg=masked_img[current_mask==False]


            total_pos_points = len(pos)
            total_neg_points = len(neg)
            pos_hist,bin_edges=np.histogram(pos, bins=20)
            neg_hist, _ = np.histogram(neg, bins=bin_edges)
            pos_hist_percent = (pos_hist / total_pos_points)
            neg_hist_percent = (neg_hist / total_neg_points)
            bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
            plt.figure(figsize=(8, 6))
            plt.bar(bin_centers, pos_hist_percent, width=bin_edges[1] - bin_edges[0], label='Contrails', alpha=0.7)
            plt.bar(bin_centers, neg_hist_percent, width=bin_edges[1] - bin_edges[0], label='Background', alpha=0.7)
            
            print("means:",np.sum(pos_hist_percent*bin_centers),np.sum(neg_hist_percent*bin_centers))
            
            plt.legend()
            # Adding labels and title
            plt.xlabel('Value')
            plt.ylabel('Counts')
            plt.title(f'Distribution of spektral indeces {i}-{j}')


            #calculating the covariance of two distributions

            covariance = np.cov(pos_hist_percent,neg_hist_percent)[0, 1]
            variance1 = np.var(pos_hist_percent)
            variance2 = np.var(neg_hist_percent)
            covariance_coefficient = covariance / np.sqrt(variance1 * variance2)
            print(f"Cor({i}-{j})",covariance_coefficient)


            # Display the plot
            plt.show()

In [ ]:
#histograms for all spectral index (one images with index 100)
idx_train_data = 100
record_id_cur = train_data_names[idx_train_data]
for i in range(8,16):
    for j in range(i+1,17):
        
        human_pixel_mask = get_human_mask(record_id_cur)
        img_test = spektral_index(i,j,record_id_cur)
        current_mask = (human_pixel_mask==1)
        masked_img = np.copy(img_test)
        pos=masked_img[current_mask]
        neg=masked_img[current_mask==False]
        

        total_pos_points = len(pos)
        total_neg_points = len(neg)
        pos_hist,bin_edges=np.histogram(pos, bins=20)
        neg_hist, _ = np.histogram(neg, bins=bin_edges)
        pos_hist_percent = (pos_hist / total_pos_points)
        neg_hist_percent = (neg_hist / total_neg_points)
        bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
        plt.figure(figsize=(8, 6))
        plt.bar(bin_centers, pos_hist_percent, width=bin_edges[1] - bin_edges[0], label='Contrails', alpha=0.7)
        plt.bar(bin_centers, neg_hist_percent, width=bin_edges[1] - bin_edges[0], label='Background', alpha=0.7)
        plt.legend()
        # Adding labels and title
        plt.xlabel('Value')
        plt.ylabel('Counts')
        plt.title(f'Distribution of spektral indeces {i}-{j}')

        # Display the plot
        plt.show()
        
        covariance = np.cov(pos_hist_percent,neg_hist_percent)[0, 1]
        variance1 = np.var(pos_hist_percent)
        variance2 = np.var(neg_hist_percent)
        covariance_coefficient = covariance / np.sqrt(variance1 * variance2)
        print(f"Cor({i}-{j})",covariance_coefficient)
        

In [ ]:
#
img=spektral_index(8,15)


plt.figure(figsize=(18, 6))
ax = plt.subplot(1, 3, 1)
ax.imshow(img)
ax.set_title('False color image')

ax = plt.subplot(1, 3, 2)
ax.imshow(human_pixel_mask, interpolation='none')
ax.set_title('Ground truth contrail mask')

ax = plt.subplot(1, 3, 3)
ax.imshow(img)
ax.imshow(human_pixel_mask, cmap='Reds', alpha=.4, interpolation='none')
ax.set_title('Contrail mask on false color image');

In [ ]:
# Individual human masks
n = human_individual_mask.shape[-1]
plt.figure(figsize=(16, 4))
for i in range(n):
    plt.subplot(1, n, i+1)
    plt.imshow(human_individual_mask[..., i], interpolation='none')

In [ ]:
pos_hist

In [ ]:
#sum of 2000 random histograms
# loop over all spectral indices
for i in range(10,15):
    for j in range(i+1,15):
        total_pos_points=0
        total_neg_points=0
        pos_hist_total=0
        neg_hist_total=0
        
        # loop over all examples
        for id_cur in range (1000):
            # load image and masks
            record_id_cur = train_data_names[id_cur]
            human_pixel_mask = get_human_mask(record_id_cur)
            img_test = spektral_index(i,j,record_id_cur)
            current_mask = (human_pixel_mask==1)
            masked_img = np.copy(img_test)
            
            # get pixels from background (neg) and contrails (pos)
            pos=masked_img[current_mask]
            neg=masked_img[current_mask==False]
            
            # make histogram for current pos -> add to total hisogram
            pos_hist,bin_edges=np.histogram(pos, bins=300, range=(0.35,1.))
            pos_hist_total=pos_hist_total+pos_hist
            
            # make histogram for current neg -> add to total hisogram
            neg_hist, _ = np.histogram(neg, bins=bin_edges)
            neg_hist_total=neg_hist_total+neg_hist
            
        # convert to percentages
        total_pos_points = pos_hist_total.sum()
        total_neg_points = neg_hist_total.sum()
        pos_hist_percent = (pos_hist_total / total_pos_points)
        neg_hist_percent = (neg_hist_total / total_neg_points)


        

        bin_centers= 0.5 * (bin_edges[1:] + bin_edges[:-1])
        hist_width=bin_edges[1]-bin_edges[0]
        plt.figure(figsize=(8, 6))
        plt.bar(bin_centers, pos_hist_percent, width=hist_width, label='Contrails', alpha=0.7)
        plt.bar(bin_centers, neg_hist_percent, width=bin_edges[1] - bin_edges[0], label='Background', alpha=0.7)

        plt.legend()
        # Adding labels and title
        plt.xlabel('Value')
        plt.ylabel('Counts')
        plt.title(f'Distribution of spektral indeces {i}-{j}')

        # Display the plot
        plt.show()

        covariance = np.cov(pos_hist_percent,neg_hist_percent)[0, 1]
        variance1 = np.var(pos_hist_percent)
        variance2 = np.var(neg_hist_percent)
        covariance_coefficient = covariance / np.sqrt(variance1 * variance2)
        print(f"Cor({i}-{j})",covariance_coefficient)


In [ ]:
pos_hist_percent